# Seasonal extreme rainfall spatial distribution

This notebook keeps the plotting code editable in-place. The expensive seasonal diagnostics are cached in NetCDF by default, including seasonal mean rainfall, seasonal 95th and 99th percentile rainfall, and seasonal 850 hPa winds.

In [ ]:
from pathlib import Path
import os
import importlib.util
import sys
import warnings

os.environ.setdefault('MPLCONFIGDIR', '/tmp/m301257_matplotlib')
os.environ.setdefault('OMP_NUM_THREADS', '1')
os.environ.setdefault('MKL_NUM_THREADS', '1')
os.environ.setdefault('OPENBLAS_NUM_THREADS', '1')
os.environ.setdefault('NUMEXPR_NUM_THREADS', '1')

import numpy as np
import xarray as xr
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import BoundaryNorm, ListedColormap

import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.mpl.ticker import LatitudeFormatter, LongitudeFormatter

warnings.filterwarnings('ignore', message='pyproj unable to set PROJ database path')

BASE_DIR = Path('/work/mh1498/m301257')
CODE_DIR = BASE_DIR / 'code_extreme_event'
SCRIPT_DIR = CODE_DIR / 'scripts'
if str(SCRIPT_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPT_DIR))
FIG_DIR = CODE_DIR / 'figures'
FIELD_DIR = CODE_DIR / 'data' / 'seasonal_fields'
FIELDS_PATH = FIELD_DIR / 'seasonal_rainfall_wind_cntl_full_domain_fields.nc'
P99_FIELDS_PATH = FIELD_DIR / 'seasonal_rainfall_p99_cntl_full_domain_fields.nc'

for path in (BASE_DIR, CODE_DIR):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

from plot_extreme_rainfall_spatial_distribution import parse_args, run_from_config

EASYXP_PATH = BASE_DIR / 'wave_tools' / 'easyxp.py'
_easyxp_spec = importlib.util.spec_from_file_location('easyxp_local', EASYXP_PATH)
if _easyxp_spec is None or _easyxp_spec.loader is None:
    raise ImportError(f'Cannot load quiver legend helper from {EASYXP_PATH}')
_easyxp_module = importlib.util.module_from_spec(_easyxp_spec)
_easyxp_spec.loader.exec_module(_easyxp_module)
simple_quiver_legend = _easyxp_module.simple_quiver_legend

print(f'Notebook working directory: {CODE_DIR}')

## Data / Cache

Leave `USE_CACHED_FIELDS=True` when only changing figure style. Set it to `False` after changing input data paths, date range, pressure level, or precipitation units. If the cache does not contain `rain_p99`, the notebook will recompute it automatically.

In [ ]:
USE_CACHED_FIELDS = True
REQUIRED_DATA_VARS = {'rain_mean', 'rain_p95', 'rain_p99', 'u850', 'v850'}

compute_args = parse_args([
    '--scheduler', 'threads',
    '--workers', 'auto',
    '--threads-per-worker', '4',
    '--wind-step', '2',
    '--dpi', '300',
])

def cache_is_complete(path):
    if not path.exists():
        return False
    with xr.open_dataset(path) as cached:
        has_vars = REQUIRED_DATA_VARS.issubset(set(cached.data_vars))
        has_seasons = 'season' in cached.coords and set(cached['season'].values.astype(str)) == {'DJF', 'MAM', 'JJA', 'SON'}
        p95_ok = has_vars and 'season' in cached['rain_p95'].dims
        p99_ok = has_vars and 'season' in cached['rain_p99'].dims
    return has_vars and has_seasons and p95_ok and p99_ok

if USE_CACHED_FIELDS and cache_is_complete(FIELDS_PATH):
    print(f'Loading cached fields: {FIELDS_PATH}')
else:
    print('Recomputing seasonal rainfall, p95/p99, and wind fields...')
    _, fields_path = run_from_config(compute_args)
    FIELDS_PATH = Path(fields_path)

ds = xr.open_dataset(FIELDS_PATH)
ds[['rain_p99']].load().to_netcdf(P99_FIELDS_PATH)
print(f'Saved P99 fields: {P99_FIELDS_PATH}')
print(ds)
ds

## Plot Controls

The values below are the main knobs for AGU-style figure tuning: figure size, fonts, line weights, color levels, quiver density, quiver legend position, and output filenames.

In [ ]:
SEASONS = ('DJF', 'MAM', 'JJA', 'SON')
PANEL_LABELS = ('(a)', '(b)', '(c)', '(d)', '(e)', '(f)')
COMPARE_LABELS = ('(a)', '(b)', '(c)', '(d)')
PERCENTILE_MAP_LABELS = ('(a)', '(b)', '(c)', '(d)', '(e)', '(f)', '(g)', '(h)')
DOMAIN = (94.0, 136.0, -14.0, 25.0)

MEAN_LEVELS = np.array([0.3, 0.75, 1.5, 3.0, 6.0, 12.0, 24.0])
P95_LEVELS = np.array([0.3, 0.75, 1.5, 3.0, 6.0, 12.0, 24.0, 48.0, 96.0])
P99_LEVELS = np.array([0.3, 0.75, 1.5, 3.0, 6.0, 12.0, 24.0, 48.0, 96.0, 144.0])
RATIO_LEVELS = np.array([1.0, 1.1, 1.2, 1.35, 1.5, 1.75, 2.0, 2.5, 3.0])
DIFF_LEVELS = np.array([0.0, 2.0, 4.0, 6.0, 8.0, 12.0, 16.0, 24.0, 32.0, 48.0])

EXTREME_COMPARE_MODE = 'ratio'  # 'ratio' for P99/P95, or 'difference' for P99-P95.

PLOT = {
    'figure_width_in': 7.15,
    'figure_height_in': 10.7,
    'compare_width_in': 7.15,
    'compare_height_in': 6.6,
    'percentile_maps_width_in': 10.2,
    'percentile_maps_height_in': 7.0,
    'font_family': 'DejaVu Sans',
    'base_fontsize': 7.5,
    'title_fontsize': 9.0,
    'panel_label_fontsize': 10.0,
    'coastline_lw': 0.65,
    'border_lw': 0.35,
    'grid_lw': 0.35,
    'map_spine_lw': 1.5,
    'box_lw': 1.0,
    'wind_step': 1,
    'wind_scale': 80,
    'wind_width': 0.0020,
    'wind_key': 5.0,
    'wind_legend_location': 'lower right',
    'cmap_name': 'cmocean_rain',
    'compare_cmap_name': 'cmocean_matter',
    'save_dpi': 600,
    'save_main_png': FIG_DIR / 'figure1_style_AGU_editable_notebook.png',
    'save_main_pdf': FIG_DIR / 'figure1_style_AGU_editable_notebook.pdf',
    'save_compare_png': FIG_DIR / 'figure_extreme_p99_p95_compare_2x2.png',
    'save_compare_pdf': FIG_DIR / 'figure_extreme_p99_p95_compare_2x2.pdf',
    'save_percentile_maps_png': FIG_DIR / 'figure_extreme_p95_p99_seasonal_maps_2x4.png',
    'save_percentile_maps_pdf': FIG_DIR / 'figure_extreme_p95_p99_seasonal_maps_2x4.pdf',
}

REGION_BOXES = {
    'PM': (99.0, 104.5, 1.0, 7.0),
    'EM': (109.0, 119.0, 0.5, 6.5),
    'WI': (99.0, 105.0, -9.0, 2.0),
    'NI': (104.5, 113.0, -5.0, 1.0),
    'SI': (104.0, 115.0, -10.0, -6.0),
    'EI': (113.0, 124.0, -5.0, 2.0),
    'NP': (120.0, 126.5, 12.0, 21.0),
    'SP': (119.0, 126.5, 5.0, 13.0),
}

In [ ]:
def set_agu_style(plot=PLOT):
    mpl.rcParams.update({
        'font.family': plot['font_family'],
        'font.size': plot['base_fontsize'],
        'axes.titlesize': plot['title_fontsize'],
        'axes.labelsize': plot['base_fontsize'],
        'xtick.labelsize': plot['base_fontsize'],
        'ytick.labelsize': plot['base_fontsize'],
        'legend.fontsize': plot['base_fontsize'],
        'figure.titlesize': plot['title_fontsize'] + 1,
        'axes.linewidth': 0.6,
        'xtick.major.width': 0.6,
        'ytick.major.width': 0.6,
        'pdf.fonttype': 42,
        'ps.fonttype': 42,
        'savefig.dpi': plot['save_dpi'],
        'savefig.bbox': 'tight',
    })


def get_colormap(name, n, low=0.08, high=0.96, pale_first=True):
    if name.startswith('cmocean_'):
        try:
            import cmocean
            base = getattr(cmocean.cm, name.replace('cmocean_', ''))
        except (ImportError, AttributeError):
            base = plt.get_cmap('YlGnBu')
    else:
        base = plt.get_cmap(name)
    colors = base(np.linspace(low, high, n))
    if pale_first:
        colors[0] = np.array([0.985, 0.985, 0.94, 1.0])
    return ListedColormap(colors)


def precipitation_cmap(levels, cmap_name='cmocean_rain'):
    return get_colormap(cmap_name, len(levels) + 1, pale_first=True)


def comparison_cmap(levels, cmap_name='cmocean_matter'):
    return get_colormap(cmap_name, len(levels) + 1, low=0.12, high=0.96, pale_first=False)


def _turn_off_cartopy_ticks(ax, labelsize=15, spine_lw=1.5):
    ax.tick_params(labelsize=labelsize, direction='out', top=False, right=False)

    from matplotlib.lines import Line2D

    try:
        ax.spines['geo'].set_visible(False)
    except KeyError:
        ax.outline_patch.set_visible(False)

    plt.draw()
    ax.xaxis.set_tick_params(top=False, which='both')
    ax.yaxis.set_tick_params(right=False, which='both')
    ax.add_artist(Line2D([0, 0], [0, 1], transform=ax.transAxes,
                         color='black', linewidth=spine_lw, clip_on=False))
    ax.add_artist(Line2D([0, 1], [0, 0], transform=ax.transAxes,
                         color='black', linewidth=spine_lw, clip_on=False))


def format_geo_axis(ax, extent=DOMAIN, plot=PLOT, left_labels=True, bottom_labels=True):
    ax.set_extent(extent, crs=ccrs.PlateCarree())
    ax.coastlines('50m', linewidth=plot['coastline_lw'])
    ax.add_feature(cfeature.BORDERS.with_scale('50m'), linewidth=plot['border_lw'])
    gl = ax.gridlines(
        crs=ccrs.PlateCarree(), draw_labels=True,
        linewidth=plot['grid_lw'], color='0.55', alpha=0.65, linestyle='--'
    )
    gl.top_labels = False
    gl.right_labels = False
    gl.left_labels = left_labels
    gl.bottom_labels = bottom_labels
    gl.xlocator = plt.FixedLocator([100, 110, 120, 130])
    gl.ylocator = plt.FixedLocator([-10, 0, 10, 20])
    gl.xformatter = LongitudeFormatter()
    gl.yformatter = LatitudeFormatter()
    gl.xlabel_style = {'size': plot['base_fontsize']}
    gl.ylabel_style = {'size': plot['base_fontsize']}
    _turn_off_cartopy_ticks(
        ax,
        labelsize=plot['base_fontsize'],
        spine_lw=plot.get('map_spine_lw', 1.5),
    )


def add_panel_label(ax, label, plot=PLOT):
    ax.text(
        0.015, 0.985, label, transform=ax.transAxes,
        ha='left', va='top', fontsize=plot['panel_label_fontsize'], fontweight='bold',
        bbox={'facecolor': 'white', 'edgecolor': 'none', 'alpha': 0.72, 'pad': 1.0},
        zorder=20,
    )


def add_region_boxes(ax, boxes=REGION_BOXES, plot=PLOT):
    for label, (lon0, lon1, lat0, lat1) in boxes.items():
        ax.plot(
            [lon0, lon1, lon1, lon0, lon0],
            [lat0, lat0, lat1, lat1, lat0],
            color='black', linewidth=plot['box_lw'], transform=ccrs.PlateCarree(), zorder=8,
        )
        ax.text(
            lon0 + 0.45, lat1 - 1.0, label,
            transform=ccrs.PlateCarree(), fontsize=plot['panel_label_fontsize'] - 1,
            fontweight='bold', ha='left', va='top', zorder=9,
        )


def add_wind_quiver(ax, u, v, plot=PLOT):
    step = 1
    q = ax.quiver(
        ds['wind_lon'][::step], ds['wind_lat'][::step],
        u.values[::step, ::step], v.values[::step, ::step],
        transform=ccrs.PlateCarree(), color='black', 
        pivot='middle',
        # width=plot['wind_width'], headwidth=3.2, headlength=4.0,
        
        # scale=2, 
        # headaxislength=3.5, zorder=7,
    )
    simple_quiver_legend(
        ax, q,
        reference_value=plot['wind_key'], unit='',
        legend_location=plot['wind_legend_location'],
        box_width=0.12, box_height=0.10, text_offset=0.020,
        font_size=plot['base_fontsize'], label_separation=0.04,
        box_facecolor='white', box_edgecolor='none', box_linewidth=0.0,
        zorder=15,
    )
    return q


def draw_rain_panel(
    ax, rain, levels, cmap, title, label,
    u=None, v=None, plot=PLOT,
    left_labels=True, bottom_labels=True,
):
    norm = BoundaryNorm(levels, cmap.N, extend='both')
    cf = ax.contourf(
        rain['lon'], rain['lat'], rain,
        levels=levels, cmap=cmap, norm=norm, extend='both',
        transform=ccrs.PlateCarree(), antialiased=False,
    )
    format_geo_axis(ax, plot=plot, left_labels=left_labels, bottom_labels=bottom_labels)
    ax.set_title(title, pad=3)
    add_panel_label(ax, label, plot=plot)
    if u is not None and v is not None:
        add_wind_quiver(ax, u, v, plot=plot)
    return cf


def percentile_field(name, season):
    return ds[name].sel(season=season)

In [ ]:
set_agu_style(PLOT)
%matplotlib inline
mean_cmap = precipitation_cmap(MEAN_LEVELS, PLOT['cmap_name'])
p95_cmap = precipitation_cmap(P95_LEVELS, PLOT['cmap_name'])

fig = plt.figure(figsize=(PLOT['figure_width_in'], PLOT['figure_height_in']))
gs = fig.add_gridspec(
    nrows=5, ncols=2,
    height_ratios=[1.0, 1.0, 0.07, 1.0, 0.07],
    hspace=0.34, wspace=0.12,
)

axes = [
    fig.add_subplot(gs[0, 0], projection=ccrs.PlateCarree()),
    fig.add_subplot(gs[0, 1], projection=ccrs.PlateCarree()),
    fig.add_subplot(gs[1, 0], projection=ccrs.PlateCarree()),
    fig.add_subplot(gs[1, 1], projection=ccrs.PlateCarree()),
    fig.add_subplot(gs[3, 0], projection=ccrs.PlateCarree()),
    fig.add_subplot(gs[3, 1], projection=ccrs.PlateCarree()),
]

mean_cf = None
for idx, season in enumerate(SEASONS):
    mean_cf = draw_rain_panel(
        axes[idx], ds['rain_mean'].sel(season=season), MEAN_LEVELS, mean_cmap,
        f'{season} mean', PANEL_LABELS[idx],
        u=ds['u850'].sel(season=season), v=ds['v850'].sel(season=season), plot=PLOT,
    )

p95_cf = draw_rain_panel(
    axes[4], percentile_field('rain_p95', 'DJF'), P95_LEVELS, p95_cmap,
    'DJF 95th percentile', PANEL_LABELS[4], plot=PLOT,
)
# add_region_boxes(axes[4], plot=PLOT)

draw_rain_panel(
    axes[5], percentile_field('rain_p95', 'JJA'), P95_LEVELS, p95_cmap,
    'JJA 95th percentile', PANEL_LABELS[5], plot=PLOT,
)

cax1 = fig.add_subplot(gs[2, :])
cbar1 = fig.colorbar(mean_cf, cax=cax1, orientation='horizontal', ticks=MEAN_LEVELS)
cbar1.ax.tick_params(length=2, width=0.6)
cbar1.set_label('precipitation (mm day$^{-1}$)')

cax2 = fig.add_subplot(gs[4, :])
cbar2 = fig.colorbar(p95_cf, cax=cax2, orientation='horizontal', ticks=P95_LEVELS)
cbar2.ax.tick_params(length=2, width=0.6)
cbar2.set_label('precipitation (mm day$^{-1}$)')

period = '1980-01-01 to 1993-12-31'
fig.suptitle(f'Seasonal rainfall and 850 hPa winds, {period}', y=0.982)
fig.subplots_adjust(top=0.945, bottom=0.055, left=0.07, right=0.98)


plt.show()
